# Loading DOEC 2000 from SGML

For the time being, the Oxford Text Archive copy of the _Dictionary of Old English_ Corpus is still [the 2000 SGML release](https://ota.bodleian.ox.ac.uk/repository/xmlui/handle/20.500.12024/2488), because OTA have not accepted new submissions since the _Dictionary of Old English_ Project last updated its corpus, which would have freed up the 2012 XML release for distribution via OTA. The 2000 release has a few complications: it stores its entity codes externally, which libraries like `lxml` do not accept, and the documents are not well-formed, so we need to make some changes to the corpus **on disk** (NB) before loading it into `lxml`.

This template assumes the official release is located in `doec2000/` with its original folder structure intact (i.e. with a subfolder `sgml-corpus/`).

In [1]:
import re
from pathlib import Path
from lxml import etree

In [2]:
# Entity matrix:
substitutions = {
    '&T;': 'Þ',
    '&t;': 'þ',
    '&D;': 'Ð',
    '&d;': 'ð',
    '&AE;': 'Æ',
    '&ae;': 'æ',
    '&E;': 'Ę',
    '&e;': 'ę',
    '&lbar;': 'ł',
    '&tbar;': '', # &thorbarslash;
    '&Aring;': 'Å',
    '&oe;': 'œ',
    '&oslash;': 'ø',
    '&bbar;': 'ƀ',
    '&auml;': 'ä',
    '&ouml;': 'ö',
    '&uuml;': 'ü',
    '&agrave;': 'à',
    '&Eacute;': 'É',
    '&eacute;': 'é',
    '&egrave;': 'è',
    '&Lambda;': 'Λ',
    '&Alpha;': 'Α',
    '&Omega;': 'Ω',
    '&Rho;': 'Ρ',
    '&Eta;': 'Η',
    '&Tau;': 'Τ',
    '&Omicron;': 'Ο',
    '&omega;': 'ω',
    '&Nu;': 'Ν',
    '&amacron;': 'ā',
    '&cmacron;': 'ċ', # c-dot is provisional in the absence of of a c-macron glyph
    '&emacron;': 'ē',
    '&imacron;': 'ī',
    '&nmacron;': '',
    '&omacron;': 'ō',
    '&pmacron;': '',
    '&qmacron;': '',
    '&rmacron;': 'r̄',
    '&supero;': 'º'
}

# Document deentification:
def deentify(doc):
    for k,v in substitutions.items():
        # Carry out replacements:
        doc = doc.replace(k, v)
    return doc

In [3]:
doec_path = Path.cwd() / 'doec2000' / 'sgml-corpus'
# DOEC SGML files from the 2000 release are not well-formed!
# So we have to repair them before lxml will open them.
# Also, entities are not internally declared, so we'll replace those too:
for file in doec_path.glob('T*sgml'):
    with open(file) as sgml:
        raw = sgml.read()
        new = re.sub(r'(<catref target="[PVG]")>', r'\g<1>/>', raw)
        deentified = deentify(new)
    with open(file, 'w') as outfile:
        outfile.write(deentified)


In [4]:
parser = etree.XMLParser(remove_blank_text=True,resolve_entities=True)
corpus = dict()
for file in doec_path.glob('T*.sgml'):
    tree = etree.parse(file, parser=parser)
    root = tree.getroot()
    short_title = root.find('.//title[@type="st"]').text
    segments = dict()
    # The 2000 corpus lacks namespaces, so we will ignore them:
    for segment in root.iter('s'):
        identifier = segment.get('n')
        segment_string = etree.tostring(segment, method='text', encoding='unicode').lstrip()
        segments[identifier] = segment_string
    corpus[short_title] = segments
        

In [5]:
corpus['Beo']

{'1': 'We Gardena in geardagum, þeodcyninga, þrym gefrunon, hu ða æþelingas ellen fremedon.',
 '4': 'Oft Scyld Scefing sceaþena þreatum, monegum mægþum, meodosetla ofteah, egsode eorlas.',
 '6': 'Syððan ærest wearð feasceaft funden, he þæs frofre gebad, weox under wolcnum, weorðmyndum þah, oðþæt him æghwylc þara ymbsittendra ofer hronrade hyran scolde, gomban gyldan.',
 '11': 'Þæt wæs god cyning.',
 '12': 'Ðæm eafera wæs æfter cenned, geong in geardum, þone god sende folce to frofre; fyrenðearfe ongeat þe hie ær drugon aldorlease lange hwile.',
 '16': 'Him þæs liffrea, wuldres wealdend, woroldare forgeaf; Beowulf wæs breme blæd wide sprang, Scyldes eafera Scedelandum in.',
 '20': 'Swa sceal geong guma gode gewyrcean, fromum feohgiftum on fæder bearme, þæt hine on ylde eft gewunigen wilgesiþas, þonne wig cume, leode gelæsten; lofdædum sceal in mægþa gehwære man geþeon.',
 '26': 'Him ða Scyld gewat to gescæphwile felahror feran on frean wære.',
 '28': 'Hi hyne þa ætbæron to brimes faroðe

String normalization and tokenization is the next step:

In [6]:
# Normalization matrix:
substitutions = {
    'ę': 'æ',
    'v': 'u',
    'j': 'i',
    #'&': 'and', # Careful! not only would "ond" be preferable in some cases, it might have to be "et" at times.
    'ł': 'uel',
    '.': '',
    ':': '',
    ';': '',
    '!': '',
    '?': '',
    '-': ''
}

# String normalization:
def normalize(txt):
    # Lowercase:
    txt = txt.lower()
    for k,v in substitutions.items():
        # Carry out replacements:
        txt = txt.replace(k, v)
    return txt


In [7]:
doec = dict()
for title,doc in corpus.items():
    document = dict()
    for identifier, segment in doc.items():
        tokens = normalize(segment).split()
        document[identifier] = tokens
    doec[title] = document

In [8]:
doec['Beo']

{'1': ['we',
  'gardena',
  'in',
  'geardagum,',
  'þeodcyninga,',
  'þrym',
  'gefrunon,',
  'hu',
  'ða',
  'æþelingas',
  'ellen',
  'fremedon'],
 '4': ['oft',
  'scyld',
  'scefing',
  'sceaþena',
  'þreatum,',
  'monegum',
  'mægþum,',
  'meodosetla',
  'ofteah,',
  'egsode',
  'eorlas'],
 '6': ['syððan',
  'ærest',
  'wearð',
  'feasceaft',
  'funden,',
  'he',
  'þæs',
  'frofre',
  'gebad,',
  'weox',
  'under',
  'wolcnum,',
  'weorðmyndum',
  'þah,',
  'oðþæt',
  'him',
  'æghwylc',
  'þara',
  'ymbsittendra',
  'ofer',
  'hronrade',
  'hyran',
  'scolde,',
  'gomban',
  'gyldan'],
 '11': ['þæt', 'wæs', 'god', 'cyning'],
 '12': ['ðæm',
  'eafera',
  'wæs',
  'æfter',
  'cenned,',
  'geong',
  'in',
  'geardum,',
  'þone',
  'god',
  'sende',
  'folce',
  'to',
  'frofre',
  'fyrenðearfe',
  'ongeat',
  'þe',
  'hie',
  'ær',
  'drugon',
  'aldorlease',
  'lange',
  'hwile'],
 '16': ['him',
  'þæs',
  'liffrea,',
  'wuldres',
  'wealdend,',
  'woroldare',
  'forgeaf',
  'beow

Now we can save the corpus in plaintext (regular and normalized) for ease of access:

In [9]:
plaintext_path = Path.cwd() / 'doec2000-plaintext'
plaintext_path.mkdir(parents=True, exist_ok=True)
for title,segments in corpus.items():
    filename = title.replace('/', '-') + '.txt'
    doc = []
    for identifier, segment in segments.items():
        doc.append(f"{identifier}: {segment}")
    target_path = plaintext_path / filename
    with open(target_path, 'w') as f:
        f.write('\n'.join(doc))


In [10]:
normalized_path = Path.cwd() / 'doec2000-bare'
normalized_path.mkdir(parents=True, exist_ok=True)
for title,segments in doec.items():
    filename = title.replace('/', '-') + '.txt'
    doc = []
    for segment in segments.values():
        doc.append(' '.join(segment))
    target_path = normalized_path / filename
    with open(target_path, 'w') as f:
        f.write('\n'.join(doc))
